In [2]:
from transformers import GPT2LMHeadModel
from mingpt.model import GPT

/root/miniconda3/envs/vila1.5/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model_config = GPT.get_default_config()
model_config.model_type = 'gpt2'
model_config.vocab_size = 50257 # openai's model vocabulary
model_config.block_size = 1024  # openai's model block_size
model = GPT(model_config)
sd = model.state_dict()

number of parameters: 209.48M


In [4]:
model_hf = GPT2LMHeadModel.from_pretrained(model_config.model_type)
sd_hf = model_hf.state_dict()

In [5]:
sd_keys = set(sd.keys())
sd_hf_keys = set(sd_hf.keys())

In [14]:
sd_hf_keys - sd_keys

{'transformer.h.1.mlp.c_fc.bias',
 'transformer.h.1.mlp.c_fc.weight',
 'transformer.h.1.mlp.c_proj.bias',
 'transformer.h.1.mlp.c_proj.weight',
 'transformer.h.11.mlp.c_fc.bias',
 'transformer.h.11.mlp.c_fc.weight',
 'transformer.h.11.mlp.c_proj.bias',
 'transformer.h.11.mlp.c_proj.weight',
 'transformer.h.3.mlp.c_fc.bias',
 'transformer.h.3.mlp.c_fc.weight',
 'transformer.h.3.mlp.c_proj.bias',
 'transformer.h.3.mlp.c_proj.weight',
 'transformer.h.5.mlp.c_fc.bias',
 'transformer.h.5.mlp.c_fc.weight',
 'transformer.h.5.mlp.c_proj.bias',
 'transformer.h.5.mlp.c_proj.weight',
 'transformer.h.7.mlp.c_fc.bias',
 'transformer.h.7.mlp.c_fc.weight',
 'transformer.h.7.mlp.c_proj.bias',
 'transformer.h.7.mlp.c_proj.weight',
 'transformer.h.9.mlp.c_fc.bias',
 'transformer.h.9.mlp.c_fc.weight',
 'transformer.h.9.mlp.c_proj.bias',
 'transformer.h.9.mlp.c_proj.weight'}

In [6]:
intersection = sd_keys.intersection(sd_hf_keys)
intersection

{'lm_head.weight',
 'transformer.h.0.attn.c_attn.bias',
 'transformer.h.0.attn.c_attn.weight',
 'transformer.h.0.attn.c_proj.bias',
 'transformer.h.0.attn.c_proj.weight',
 'transformer.h.0.ln_1.bias',
 'transformer.h.0.ln_1.weight',
 'transformer.h.0.ln_2.bias',
 'transformer.h.0.ln_2.weight',
 'transformer.h.0.mlp.c_fc.bias',
 'transformer.h.0.mlp.c_fc.weight',
 'transformer.h.0.mlp.c_proj.bias',
 'transformer.h.0.mlp.c_proj.weight',
 'transformer.h.1.attn.c_attn.bias',
 'transformer.h.1.attn.c_attn.weight',
 'transformer.h.1.attn.c_proj.bias',
 'transformer.h.1.attn.c_proj.weight',
 'transformer.h.1.ln_1.bias',
 'transformer.h.1.ln_1.weight',
 'transformer.h.1.ln_2.bias',
 'transformer.h.1.ln_2.weight',
 'transformer.h.10.attn.c_attn.bias',
 'transformer.h.10.attn.c_attn.weight',
 'transformer.h.10.attn.c_proj.bias',
 'transformer.h.10.attn.c_proj.weight',
 'transformer.h.10.ln_1.bias',
 'transformer.h.10.ln_1.weight',
 'transformer.h.10.ln_2.bias',
 'transformer.h.10.ln_2.weight',
 

In [7]:
cp_keys = list(sd_hf_keys - intersection)
cp_keys

['transformer.h.3.mlp.c_fc.weight',
 'transformer.h.3.mlp.c_proj.bias',
 'transformer.h.7.mlp.c_fc.bias',
 'transformer.h.9.mlp.c_fc.weight',
 'transformer.h.7.mlp.c_fc.weight',
 'transformer.h.5.mlp.c_fc.bias',
 'transformer.h.5.mlp.c_fc.weight',
 'transformer.h.11.mlp.c_proj.weight',
 'transformer.h.9.mlp.c_fc.bias',
 'transformer.h.7.mlp.c_proj.weight',
 'transformer.h.11.mlp.c_fc.bias',
 'transformer.h.1.mlp.c_fc.weight',
 'transformer.h.11.mlp.c_proj.bias',
 'transformer.h.9.mlp.c_proj.bias',
 'transformer.h.9.mlp.c_proj.weight',
 'transformer.h.1.mlp.c_proj.weight',
 'transformer.h.1.mlp.c_proj.bias',
 'transformer.h.3.mlp.c_proj.weight',
 'transformer.h.3.mlp.c_fc.bias',
 'transformer.h.1.mlp.c_fc.bias',
 'transformer.h.5.mlp.c_proj.bias',
 'transformer.h.11.mlp.c_fc.weight',
 'transformer.h.7.mlp.c_proj.bias',
 'transformer.h.5.mlp.c_proj.weight']

In [8]:
def generate_key_names(original):
    return [original.replace('mlp', f'mlp.experts.{i}') for i in range(4)]

In [9]:
key = generate_key_names(cp_keys[0])

In [10]:
for old_key in cp_keys:
    for new_key in generate_key_names(old_key):
        sd_hf[new_key] = sd_hf[old_key]
    sd_hf.pop(old_key)

In [11]:
sd_hf_new_keys = set(sd_hf.keys())

In [12]:
sd_hf_new_keys - sd_hf_new_keys.intersection(sd_keys)

set()